# **Simple RAG Demo — HR Policy Assistant**

RAG stands for **Retrieval-Augmented Generation**.

It is an AI architecture that allows an LLM to retrieve relevant information from your own data before generating an answer.

```python
             HR Policy Handbook
                     │
                     ▼
              ┌──────────────┐
              │   Document   │
              └──────┬───────┘
                     │
                     ▼
              Split into chunks
                     │
          ┌──────────┼──────────┐
          ▼          ▼          ▼
       Chunk 1    Chunk 2    Chunk 3   ...
          │          │          │
          ▼          ▼          ▼
     Embedding    Embedding   Embedding
          │          │          │
          ▼          ▼          ▼
      Vector 1     Vector 2    Vector 3
          │          │          │
          └──────────┼──────────┘
                     ▼
              ┌──────────────┐
              │ Vector Store │
              │      +       │
              │    FAISS     │
              └──────┬───────┘
                     │
                     │
              User asks question
                     │
                     ▼
          "How many sick leaves
             can I take?"
                     │
                     ▼
              Create embedding
              for the question
                     │
                     ▼
              Search Vector Store
                     │
                     ▼
          Find most relevant chunks
                     │
              ┌──────┼──────┐
              ▼      ▼      ▼
           Chunk 7  Chunk 12  Chunk 18
              │      │      │
              └──────┼──────┘
                     ▼
          Relevant context + question
                     │
                     ▼
              ┌──────────────┐
              │     LLM      │
              │ Groq API Key │
              └──────┬───────┘
                     │
                     ▼
              Final Answer
                     │
                     ▼
      "According to the HR policy,
       employees are entitled to..."
```

What we use in this notebook:

Data: data/hr_policy.txt (a sample HR policy document)

Embeddings: Jina AI Embeddings Provider

Vector store: FAISS

LLM: Groq (fast + free-tier friendly)

Framework: LangChain (create_agent)

Before running: make sure the notebook's kernel is set to the ragenv virtual environment (top-right corner of VS Code / Jupyter), and that a .env file with GROQ_API_KEY and JINA_API_KEY exists in this folder.

### **Step 1 — Import everything we need**

We import all the tools upfront so it's clear what's being used and where it comes from.

In [493]:
import os
from dotenv import load_dotenv

# Langchain imports
from langchain_community.document_loaders import TextLoader #TextLoader is a specialized document loader used to read plain text files (.txt) and transform their contents into standard LangChain Document objects.
from langchain_text_splitters import RecursiveCharacterTextSplitter #RecursiveCharacterTextSplitter is LangChain's recommended tool for splitting long text documents into smaller, semantically meaningful chunks.
from langchain_community.embeddings import JinaEmbeddings #JinaEmbeddings is a class that provides access to Jina's pre-trained embeddings models, which can be used to convert text into vector representations for semantic search and retrieval tasks.
from langchain_community.vectorstores import FAISS #FAISS is a library for efficient similarity search and clustering of dense vectors. In LangChain, it is used to create vector stores that allow for fast retrieval of documents based on their embeddings.
from langchain_groq import ChatGroq #ChatGroq is a class that provides an interface to interact with Groq's language model for generating responses based on input queries and context.
from langchain.agents import create_agent #create_agent is a function that allows you to create an agent that can interact with a language model and perform tasks based on user input and context.


In [494]:
load_dotenv()  # Load environment variables from .env file

True

In [495]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

print("ENVIRONMENT VARIABLES ARE LOADED SUCCESSFULLY")

ENVIRONMENT VARIABLES ARE LOADED SUCCESSFULLY


# **DATA INGESTION PIPELINE**

## **(A) LOADING OUR DATA**

In [496]:
from langchain_community.document_loaders import TextLoader

DATA_DIR = os.path.join("hr_policy", "policy_documents.txt")  # Directory containing policy documents

In [497]:
loader = TextLoader(DATA_DIR, encoding="utf-8")  # Load text files with UTF-8 encoding
documents = loader.load()  # Load the documents into a list

print(f"Loaded {len(documents)} documents from {DATA_DIR}")  # Print the number of loaded documents
print(documents)

Loaded 1 documents from hr_policy\policy_documents.txt
[Document(metadata={'source': 'hr_policy\\policy_documents.txt'}, page_content="COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nPurpose:\nThe Leave Policy defines the types of leave available to employees, eligibility requirements, approval procedures, and rules for managing unused leave.\nPolicy:\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nLeave should be approved by the employee's reporting manager before the employee takes the leave.\nUnused annual leave may be carried forward to the following calendar year, up to a maximum of 5 days.\nAny annual leave exceeding the permitted carry-forward limit may expire at the end of the applicable period.\nSick leave is separate from annual leave.\nEmployees are entitled to 10 paid sick days per year.\nEmployee

### **LANGCHAIN DOCUMENTS**

Langchain processes everything in form of documents

DOCUMENTS :

PAGE CONTENT -- the actual data

METADATA - extra information about the data

In [498]:
len(documents)

1

In [499]:
print(documents[0].page_content)

COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
Purpose:
The Leave Policy defines the types of leave available to employees, eligibility requirements, approval procedures, and rules for managing unused leave.
Policy:
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Leave should be approved by the employee's reporting manager before the employee takes the leave.
Unused annual leave may be carried forward to the following calendar year, up to a maximum of 5 days.
Any annual leave exceeding the permitted carry-forward limit may expire at the end of the applicable period.
Sick leave is separate from annual leave.
Employees are entitled to 10 paid sick days per year.
Employees should notify their manager as soon as reasonably possible when they are unable to work due to illness.
A medical certificate is required when sick

In [500]:
documents[0].metadata

{'source': 'hr_policy\\policy_documents.txt'}

In [501]:
print(f"Total characters in the document: {len(documents[0].page_content)}")

Total characters in the document: 25773


Upto here data loading is completed sucessfully.

It is important to know the size of the data. So, that we can easy divided into chucks or evenly disturbuted

## **(B) SPLITTING OUR DATA** 

In [502]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1450,  # Maximum number of characters in each chunk
    chunk_overlap=70   # Number of characters to overlap between chunks
)

chunks = splitter.split_documents(documents)  # Split the documents into chunks

print(chunks)
print(f"Total chunks created: {len(chunks)}")  # Print the number of chunks created

[Document(metadata={'source': 'hr_policy\\policy_documents.txt'}, page_content="COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nPurpose:\nThe Leave Policy defines the types of leave available to employees, eligibility requirements, approval procedures, and rules for managing unused leave.\nPolicy:\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nLeave should be approved by the employee's reporting manager before the employee takes the leave.\nUnused annual leave may be carried forward to the following calendar year, up to a maximum of 5 days.\nAny annual leave exceeding the permitted carry-forward limit may expire at the end of the applicable period.\nSick leave is separate from annual leave.\nEmployees are entitled to 10 paid sick days per year.\nEmployees should notify their manager as soon as reasonably pos

##### Now each splitted chunk is a document - So Document contains page_content & metadata

In [503]:
len(chunks)

21

In [504]:
print(chunks[0].page_content)  # Print the content of the first chunk

COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
Purpose:
The Leave Policy defines the types of leave available to employees, eligibility requirements, approval procedures, and rules for managing unused leave.
Policy:
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Leave should be approved by the employee's reporting manager before the employee takes the leave.
Unused annual leave may be carried forward to the following calendar year, up to a maximum of 5 days.
Any annual leave exceeding the permitted carry-forward limit may expire at the end of the applicable period.
Sick leave is separate from annual leave.
Employees are entitled to 10 paid sick days per year.
Employees should notify their manager as soon as reasonably possible when they are unable to work due to illness.
A medical certificate is required when sick

In [505]:
print(chunks[1].page_content)

2. WORK FROM HOME POLICY
Purpose:
The Work From Home Policy defines the conditions under which employees may work remotely and establishes expectations regarding availability, productivity, and communication.
Policy:
Employees may work from home for up to 2 days per week, subject to manager approval.
Work-from-home requests should be submitted through the company's designated process.
Managers may approve or decline requests based on business requirements, team availability, workload, and employee responsibilities.
Fully remote work arrangements require written approval from the department head.
Employees working from home must remain reachable during core working hours of 10:00 AM to 4:00 PM.
Employees must attend scheduled meetings and remain available through approved communication channels.
Employees are expected to maintain normal productivity and meet assigned deadlines while working remotely.
Employees must maintain a suitable and secure working environment.
Confidential company

In [506]:
print(chunks[2].page_content)

3. PROBATION PERIOD
Purpose:
The Probation Period Policy defines the initial evaluation period for newly hired employees and establishes expectations during probation.
Policy:
All new employees undergo a probation period of 3 months from their date of joining.
The probation period is intended to evaluate the employee's performance, conduct, skills, attendance, and suitability for the position.
During probation, employees are not eligible for paid leave.
Employees may take unpaid leave in emergency situations, subject to manager approval.
Employees must follow all company policies during the probation period.
Managers should provide appropriate feedback regarding the employee's performance.
Performance is formally reviewed at the end of the probation period.
Based on the review, the company may confirm the employee's employment.
Where performance does not meet expectations, the company may take appropriate action according to applicable company procedures.
Employees remain subject to th

In [507]:
print(chunks[3].page_content)

4. NOTICE PERIOD
Purpose:
The Notice Period Policy defines the resignation notice requirements for employees and the applicable procedures during the separation process.
Policy:
Employees who wish to resign must provide formal written notice to the company.
The standard notice period is 30 days.
Employees serving their probation period have a reduced notice period of 15 days.
The notice period normally begins from the date the resignation is formally accepted or recorded according to company procedures.
Employees are expected to continue performing their assigned responsibilities during the notice period.
Employees should complete knowledge transfer and transition activities before their last working day.
Company assets must be returned according to the exit process.
The company may waive the notice period at its discretion.
Any approved early release or notice-period waiver must be documented by the appropriate authority.
Full and final settlement is processed within 45 days of the la

In [508]:
print(chunks[4].page_content)

5. REIMBURSEMENT POLICY
Purpose:
The Reimbursement Policy defines the process for claiming approved business expenses incurred by employees while performing official company activities.
Policy:
Employees may claim reimbursement for approved business expenses.
Eligible expenses may include:
Business travel
Client meals
Internet expenses used for official work
Other approved business-related expenses
Expenses must be directly related to legitimate business activities.
Employees must obtain required approval before incurring expenses where prior approval is required.
All reimbursement claims must be supported by valid bills, receipts, or other required documentation.
Claims must be submitted within 30 days of the expense.
Employees are responsible for ensuring that submitted information is accurate.
Duplicate, personal, or unsupported expenses may be rejected.
The reporting manager must review and approve the reimbursement claim.
Claims are processed within 10 working days after reporting

In [509]:
print(chunks[5].page_content)

6. CODE OF CONDUCT
Purpose:
The Code of Conduct Policy establishes standards for professional, ethical, respectful, and responsible behavior in the workplace.
Policy:
Employees are expected to maintain professionalism and respect in all workplace interactions.
Employees must treat colleagues, customers, business partners, and other individuals fairly and respectfully.
Harassment, discrimination, bullying, intimidation, or other forms of workplace misconduct are prohibited.
Employees must comply with applicable company policies and workplace procedures.
Employees are expected to protect confidential company and customer information.
Employees must avoid behavior that could damage the company's professional environment or reputation.
Conflicts of interest should be disclosed to the appropriate manager or HR representative.
Employees must cooperate with legitimate internal investigations.
Violations of the Code of Conduct may result in disciplinary action.
Depending on the nature and seve

In [510]:
print(chunks[6].page_content)

7. HOLIDAYS
Purpose:
The Holiday Policy defines the company's public holiday schedule and the treatment of employees who are required to work on designated holidays.
Policy:
The company observes 12 public holidays every year.
The official holiday calendar is published by HR at the start of each year.
Employees should refer to the official HR holiday calendar for the applicable holiday dates.
Holiday schedules may vary depending on business location or applicable requirements.
Employees should plan their leave and work activities around the published holiday calendar.
Employees required to work on a public holiday are eligible for compensatory leave.
Compensatory leave should be requested and used according to the company's applicable procedures.
Managers should ensure that employees required to work on holidays are appropriately informed.
The official HR holiday calendar takes precedence over informal or unofficial holiday lists.


In [511]:
print(chunks[7].page_content)

8. EXIT POLICY
Purpose:
The Exit Policy establishes the procedures employees must follow when leaving the company through resignation or termination.
Policy:
Employees leaving the company must complete the required separation and clearance process.
The clearance process involves the IT, Finance, and HR departments.
Employees must return company property, including laptops, access cards, documents, and other assigned assets.
IT is responsible for completing the relevant IT-asset and system-access clearance.
Finance is responsible for reviewing outstanding financial obligations, reimbursements, advances, or other applicable matters.
HR is responsible for completing employment records and separation documentation.
Employees should complete required knowledge transfer before their last working day.
Employees must complete all required clearance activities before or around their last working day according to company procedures.
Full and final settlement includes applicable pending payments,

In [512]:
print(chunks[8].page_content)

9. ATTENDANCE POLICY
Purpose:
The Attendance Policy defines the company's expectations regarding employee attendance, punctuality, attendance recording, and regular presence during scheduled working hours.
Policy:
Employees are expected to report to work according to their assigned work schedule.
Employees must record their attendance using the company's designated attendance system, HR portal, or access-control system.
Employees are expected to be punctual and ready to begin work at their scheduled start time.
Repeated late arrivals, early departures, or unexplained absences may be considered attendance violations.
Employees who are unable to report to work must notify their manager as early as possible.
Absences without prior notification or approval may be treated as unauthorized absence.
Managers are responsible for monitoring attendance and addressing recurring attendance issues.
Employees must accurately record their own attendance and must not record attendance on behalf of anot

In [513]:
print(chunks[19].page_content)

Records that have reached the end of their approved retention period may be securely disposed of in accordance with company procedures.
Electronic records should be securely deleted using approved methods when disposal is authorized. Physical records containing confidential or sensitive information should be securely destroyed using approved methods such as secure shredding.
Employees who receive a records-preservation or legal-hold notice must follow the instructions provided and cooperate with the relevant Legal, Compliance, HR, IT, or other authorized teams.
Managers are responsible for ensuring that employees maintain records required for their business functions.
HR is responsible for maintaining employee-related records according to applicable retention requirements and company procedures.
Finance is responsible for maintaining financial and accounting records according to applicable legal, tax, audit, and company requirements.


## **(C) Embedding our data**

In [514]:
from langchain_community.embeddings import JinaEmbeddings

embeddings = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")

print("Embedded Model ready the name is: ", embeddings.model_name)

Embedded Model ready the name is:  jina-embeddings-v2-base-en


## **(D) Storing our data in vector database**

Faiss (Facebook AI Similarity Search) is an open-source library created by Meta for fast similarity search and clustering of dense vectors.


Key Features

1. High Speed: Finds similar multimedia documents or vector embeddings much faster than traditional search engines.
2. Massive Scale: Handles datasets ranging from a few million to over a billion items.
3. Hardware Support: Written in C++ with Python wrappers and includes GPU acceleration for heavy workloads.
4. Flexible Indexing: Offers methods like exact brute-force search (IndexFlatL2) and scalable approximate search (IndexIVF)


Common use cases:

1. Recommendation System: Finding similar user profile or item embeddings quickly.
2. Imformation Retrieval: Powering semantic search engines and RAG pipelines.
3. Multimedia Search: Retrieving visually simlar images, video, or audio tracks.

In [515]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, embeddings)  # Create a FAISS vector store from the document chunks and embeddings

print("Chunks are embedded and stored in the FAISS vector store successfully.", "Total number of vectors in the store:", vectorstore.index.ntotal)  # Print the total number of vectors in the FAISS index

Chunks are embedded and stored in the FAISS vector store successfully. Total number of vectors in the store: 21


In [516]:
test_sample_query = "What is the company's policy on remote work?"

## Similarity search using the FAISS vector store

top_matches = vectorstore.similarity_search(test_sample_query, k=2)  # Retrieve the top 3 most similar chunks based on the query

print(f"Query: {test_sample_query}\n")

for i,match in enumerate(top_matches, start=1):
    print(f"Match {i}:")
    print(match.page_content)
    print()  # Separator for readability

Query: What is the company's policy on remote work?

Match 1:
2. WORK FROM HOME POLICY
Purpose:
The Work From Home Policy defines the conditions under which employees may work remotely and establishes expectations regarding availability, productivity, and communication.
Policy:
Employees may work from home for up to 2 days per week, subject to manager approval.
Work-from-home requests should be submitted through the company's designated process.
Managers may approve or decline requests based on business requirements, team availability, workload, and employee responsibilities.
Fully remote work arrangements require written approval from the department head.
Employees working from home must remain reachable during core working hours of 10:00 AM to 4:00 PM.
Employees must attend scheduled meetings and remain available through approved communication channels.
Employees are expected to maintain normal productivity and meet assigned deadlines while working remotely.
Employees must maintain a

### **Tool**

In [556]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})  # Create a retriever from the FAISS vector store

def search_hr_policy(question: str) -> str:
    """
    Search the HR policy documents for relevant information based on the provided question. 
    """ # Documentation string explaining the purpose of the function
    matches = retriever.invoke(question)
    return "/n/n".join(chunk.page_content for chunk in matches)  # Join the content of the matches with newline separators

# **DATA RETRIEVAL PIPELINE**

#### **LLMs**

Large Language Models(LLMs) is a type of artificial intelligence (AI) program trained on massive amounts of text to understand, summarize, generate, and translate human language.

How LLMs Work

1. Transformer Architecture: Most LLMs use a deep learning framework called a transformer, which processes words and contextual relationships in parallel.
2. Next-Token Prediction: They function as large-scale statistical prediction machines that break text into smaller units (tokens) and repeatedly guess the most probable next word or token.
3. Training Stages: They go through initial self-supervised pre-training on vast data pools, followed by fine-tuning to safely follow user instructions.

Common Use Cases

1. Text Generation: Drafting articles, emails, stories, or legal clauses.
2. Summarization: Condensing long documents or transcripts into key points.
3. Translation: Converting text smoothly between multiple human languages.
4. Coding: Writing, debugging, and explaining computer programming code.

In [557]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",  # Specify the Groq language model to use
    temperature=0.9,  # Set the temperature for response generation (controls randomness) - Creativity of the response. Lower values make the output more deterministic.
)

llm.model_name

'openai/gpt-oss-120b'

In [558]:
test_sample_response = llm.invoke("Learning RAG with LangChain and Groq is really hard? Give me the response in 1 sentence. Reply in a joke manner")

test_sample_response.content

'Learning RAG with LangChain and Groq is about as easy as convincing a cat to write clean code—sure, it’ll stare at the screen, but the purr‑formance will still be a mystery!'

#### **AI Agent**

AI agent is an autonomous software system powered by artificial intelligence that can perceive its environment, plan steps, use external tools, and execute multi-step workflows to achieve a specific goal with minimal human oversight.

Core Components

1. The Brain (LLM): Large language models process inputs, handle reasoning, and decide which actions or tool calls to make next.
2. Tool Integration: APIs, web browsers, and software functions act as the agent's hands to retrieve live data or perform tasks.
3. Memory: Short-term context windows and long-term vector databases retain past interactions and project states over time.
4. Autonomous Loop: The system reasons, executes an action, observes the outcome, and loops until the objective is met.

Common Types

1. Reactive Agents: Respond directly to current environment states using simple rules.
2. Goal-Based Agents: Take actions specifically directed toward achieving a defined outcome.
3. Utility-Based Agents: Choose the option that provides the highest value or best result among alternatives.
4. Learning Agents: Adapt their future behavior and improve performance based on past feedback.


An agent is a model calling tools in a loop until a given task is complete.

```python
                    ┌──────────────┐
                    │     User     │
                    └──────┬───────┘
                           ↓
                    ┌──────────────┐
                    │     Agent    │
                    │     + LLM    │
                    └──────┬───────┘
                           ↓
                 ┌────────────────────┐
                 │  Which tool?       │
                 └─────────┬──────────┘
                           ↓
          ┌────────────────┼────────────────┐
          ↓                ↓                ↓
     Calculator       RAG Search        Weather API
          │                │                │
          └────────────────┼────────────────┘
                           ↓
                    ┌──────────────┐
                    │     Agent    │
                    └──────┬───────┘
                           ↓
                         Answer
```

A harness is everything around that loop: the prompt, the tools, and any middleware that shapes the model’s behavior.

Agent = Model + Harness

The job of a harness: get the model the right context at the right time for the given task.

### **How many tools can you create?**

**There is no fixed limit.** LangChain lets you attach as many tools as you want to an agent - 1, 5, 20 or 100. `create_agent(model, tools=[...])` simply takes a list, and that list can be any length.

The real limit is **not the framework, it is the model**:

| Factor | Why it matters |
|---|---|
| Context window | Every tool's name + docstring + JSON schema is injected into the prompt. Too many tools eat into the space left for the retrieved policy text. |
| Tool-selection accuracy | Models get confused when many tools have overlapping descriptions. With 20 similar tools the agent picks the wrong one far more often than with 5 clearly distinct ones. |
| Latency and cost | More tools means a bigger prompt on **every** turn of the agent loop, not just the first one. |
| Groq free tier | Rate limits (TPM / RPM) apply per request, so a heavy multi-tool loop can get throttled. |

**Rule of thumb for this HR Policy Assistant: 4-6 tools is the sweet spot.** That is enough to cover the realistic question types - search the handbook, list the sections, do leave arithmetic, get today's date, escalate - without confusing the model.

### **What kind of tools make sense for an HR Policy Assistant?**

Tools fall into a few families. Here is the mapping for this project:

| Tool family | What it does | Example in this notebook |
|---|---|---|
| **Retrieval (RAG)** | Searches the vector store and returns relevant policy chunks. This is the heart of the whole notebook. | `hr_policy_search` |
| **Deterministic lookup** | Returns fixed, structured data with no LLM guessing. More reliable than semantic search for "what policies exist?". | `policy_catalog` |
| **Utility / computation** | Pure functions the LLM is genuinely bad at - math, dates, counting. | `calculate_leave_days`, `current_date` |
| **Text extraction** | Pulls numbers, day-counts and deadlines out of policy text using regex. | `extract_policy_numbers` |
| **Action / write** | Tools with side effects - log a ticket, send a notification, escalate. **These need human approval in a real HR system.** | `escalate_to_hr` |
| **External API (optional)** | Live data from outside - holiday calendar, HRIS, Slack. Left out here so the notebook stays free-tier friendly and offline. | `holiday_calendar_api`, `leave_balance_lookup` |

> Key principle: **the LLM decides *which* tool to call, the tool does the actual work.** Notice that retrieval itself is a tool - so the agent reads the policy only when the question actually needs it, instead of stuffing the whole handbook into every prompt.

In [562]:
from langchain.agents import create_agent

In [568]:
hr_assistant = create_agent(
    model=llm,  # Use the Groq language model for the agent
    tools=[search_hr_policy],  # Provide the search_hr_policy tool for the agent to use
    system_prompt="""
    You are a friendly HR assistant. 
    Always use the search_hr_policy tool to look up
    facts before answering questions about HR policies.
    If you cannot find the answer, respond with "I don't know".
    instead of guessing.
    """
)

print("HR Assistant agent is created successfully. You can now interact with it to answer HR policy-related questions.")

HR Assistant agent is created successfully. You can now interact with it to answer HR policy-related questions.


In [569]:
def ask_hr_assistant(question: str) -> str:
    """Send a question to the RAG agent and print a nicely formatted answer."""
    print("=" * 60)
    print("QUESTION:", question)
    print("-" * 60)

    #response = hr_assistant.invoke({"messages": [{"role": "user", "content": question}]})
    answer = response["messages"][-1].content

    print("ANSWER:", answer)
    print("=" * 60)
    print()
    return answer

In [570]:
response = hr_assistant.invoke(
    {
        "messages": [{
            "role": "user",
            "content": "What is the company's policy on remote work?"
        }]
    }
)

In [579]:
response

{'messages': [HumanMessage(content="What is the company's policy on remote work?", additional_kwargs={}, response_metadata={}, id='bc754c04-52bb-41de-8220-3b8b606aedbf'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "We need to answer about company's policy on remote work. Must use search_hr_policy tool. Let's call it.", 'tool_calls': [{'id': 'fc_d3ba0d13-c32e-4b9a-89f2-e0b0df428457', 'function': {'arguments': '{"question":"What is the company\'s policy on remote work?"}', 'name': 'search_hr_policy'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 191, 'total_tokens': 249, 'completion_time': 0.122356404, 'completion_tokens_details': {'reasoning_tokens': 23}, 'prompt_time': 0.016187116, 'prompt_tokens_details': None, 'queue_time': 0.051306683, 'total_time': 0.13854352}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_420c059a6a', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': 

In [590]:
print(response["messages"][-1].content)

**Company Remote‑Work (Work‑From‑Home) Policy**

- **Eligibility & Frequency**  
  - Employees may work from home up to **2 days per week**, but only after receiving **manager approval**.

- **Request Process**  
  - Requests must be submitted through the company’s designated work‑from‑home request process.  
  - Managers approve or decline based on business needs, team workload, and the employee’s responsibilities.

- **Fully Remote Arrangements**  
  - A completely remote (full‑time) work arrangement requires **written approval from the department head**.

- **Availability & Communication**  
  - While working remotely, employees must be reachable during **core hours 10:00 AM – 4:00 PM**.  
  - They must attend scheduled meetings and stay available via the approved communication channels.

- **Productivity & Performance**  
  - Remote employees are expected to maintain normal productivity, meet deadlines, and fulfill all job responsibilities.

- **Work Environment & Security**  
  - 

SYSTEM MESSAGE

HUMAN MESSAGE

AI MESSAGE

In [591]:
response["messages"][-1].content

'**Company Remote‑Work (Work‑From‑Home) Policy**\n\n- **Eligibility & Frequency**  \n  - Employees may work from home up to **2 days per week**, but only after receiving **manager approval**.\n\n- **Request Process**  \n  - Requests must be submitted through the company’s designated work‑from‑home request process.  \n  - Managers approve or decline based on business needs, team workload, and the employee’s responsibilities.\n\n- **Fully Remote Arrangements**  \n  - A completely remote (full‑time) work arrangement requires **written approval from the department head**.\n\n- **Availability & Communication**  \n  - While working remotely, employees must be reachable during **core hours\u202f10:00\u202fAM\u202f–\u202f4:00\u202fPM**.  \n  - They must attend scheduled meetings and stay available via the approved communication channels.\n\n- **Productivity & Performance**  \n  - Remote employees are expected to maintain normal productivity, meet deadlines, and fulfill all job responsibilities

## **Vector Store**

Vector stores are specialized databases that hold data as mathematical number arrays called embeddings, allowing AI models to search for information by conceptual meaning rather than exact keywords.

In-Memory Vector Stores:

1. What it is: Data lives entirely in the computer's active RAM while the application runs.
2. Key traits: Extremely fast search speeds because data is read directly from memory.
3. Drawback: Temporary; when the app or server restarts, all stored data disappears unless saved elsewhere.
4. Best use: Testing ideas, rapid prototyping, or small temporary tasks.

Persistent Vector Stores:

1. What it is: Data is saved to a physical disk or local file system.
2. Key traits: Data survives application restarts and system reboots.
3. Drawback: Slightly slower read/write speeds compared to pure RAM.
4. Best use: Local applications, small production tools, or projects needing data to stay saved on a single machine.

Cloud Memory (Cloud-Native Vector Stores): 

1. What it is: Fully managed vector databases hosted on remote cloud infrastructure (like Pinecone or cloud storage buckets).
2. Key traits: Highly scalable, accessible from anywhere via APIs, and managed by a provider so you do not maintain the physical hardware.
3. Drawback: Requires an internet connection, has usage costs, and adds network latency.
4. Best use: Large-scale commercial AI apps, production environments, and multi-user systems handling millions of data points.